# Post-Hoc Semantic Evaluation

This notebook performs semantic evaluation on already-collected VLM results.

## Purpose
- Load results from `fast_parallel_evaluation.ipynb` runs
- Run expensive Glider evaluations separately
- Compute semantic F1 scores using Molmo-style atomic statement comparison
- Add Glider rubric scores with reasoning

## Advantages
- Decouple data collection from evaluation
- Run evaluation on subsets (save cost/time)
- Re-evaluate with different parameters without re-running VLM inference
- Process results in smaller batches to avoid timeouts

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm
import json
import time
from typing import Dict, List, Optional

# Import evaluation utilities
import sys
sys.path.append('/mnt/user-data/uploads')
from evaluation import GliderEvaluator
import fast_parallel_evaluation_utils as eval_utils

## 2. Configuration

In [ ]:
# === INPUT CONFIGURATION ===
# Specify the run to evaluate
RUN_ID = "exp_20250127_123456"  # Change this to your actual run ID
INPUT_DIR = Path(f"./experiment_data/runs/{RUN_ID}")

# Or use the latest run automatically
USE_LATEST_RUN = True
if USE_LATEST_RUN:
    run_dirs = sorted(Path("./experiment_data/runs").glob("exp_*"))
    if run_dirs:
        INPUT_DIR = run_dirs[-1]
        RUN_ID = INPUT_DIR.name
        print(f"📂 Using latest run: {RUN_ID}")
    else:
        raise FileNotFoundError("No experiment runs found")

# === OUTPUT CONFIGURATION ===
OUTPUT_SUFFIX = f"_semantic_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = INPUT_DIR / "semantic_evaluation"
OUTPUT_DIR.mkdir(exist_ok=True)

# === EVALUATION CONFIGURATION ===
GLIDER_PORT = 8805
REQUEST_TIMEOUT = 180  # seconds

# Control what evaluations to run
ENABLE_SEMANTIC_F1 = True   # Molmo-style atomic statement comparison
ENABLE_GLIDER_RUBRIC = True  # LLM-as-judge rubric scoring

# Sampling strategy
SAMPLE_STRATEGY = "all"  # Options: "all", "random", "per_model", "per_config"
N_SAMPLES_TOTAL = 1000   # If using random sampling
N_SAMPLES_PER_MODEL = 200  # If using per_model sampling
N_SAMPLES_PER_CONFIG = 50  # If using per_config sampling

# Processing configuration
BATCH_SIZE = 10  # Process in small batches to avoid overwhelming Glider
SAVE_INTERMEDIATE = True  # Save after each batch

# Filter options
FILTER_GT_TYPES = None  # e.g., ["freeform", "exact"] or None for all
FILTER_MODELS = None     # e.g., ["gemma-3-27b"] or None for all
FILTER_CONFIGS = None    # e.g., ["docvqa", "chartqa"] or None for all

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Semantic F1: {ENABLE_SEMANTIC_F1}")
print(f"Glider Rubric: {ENABLE_GLIDER_RUBRIC}")
print(f"Sample strategy: {SAMPLE_STRATEGY}")

# Configure evaluation utilities
eval_utils.configure(
    request_timeout=REQUEST_TIMEOUT,
    evaluator_port=GLIDER_PORT,
)

## 3. Load Results

In [ ]:
# Load the combined results file
results_file = INPUT_DIR / "all_results.parquet"

if not results_file.exists():
    print(f"❌ Results file not found: {results_file}")
    print("\nAvailable files:")
    for f in INPUT_DIR.glob("*.parquet"):
        print(f"  - {f.name}")
    raise FileNotFoundError("Combined results file not found")

print(f"Loading results from: {results_file}")
df_all = pd.read_parquet(results_file)

print(f"\n📊 Loaded {len(df_all):,} records")
print(f"   Configs: {df_all['source_config'].nunique()}")
print(f"   Models: {df_all['model_name'].nunique()}")
print(f"   Samples: {df_all['sample_id'].nunique()}")

# Show available columns
print(f"\n📋 Available columns:")
for col in sorted(df_all.columns):
    print(f"   - {col}")

## 4. Filter & Sample Data

In [ ]:
# Apply filters
df_filtered = df_all.copy()

if FILTER_GT_TYPES:
    df_filtered = df_filtered[df_filtered['ground_truth_type'].isin(FILTER_GT_TYPES)]
    print(f"Filtered to ground truth types: {FILTER_GT_TYPES}")

if FILTER_MODELS:
    df_filtered = df_filtered[df_filtered['model_name'].isin(FILTER_MODELS)]
    print(f"Filtered to models: {FILTER_MODELS}")

if FILTER_CONFIGS:
    df_filtered = df_filtered[df_filtered['source_config'].isin(FILTER_CONFIGS)]
    print(f"Filtered to configs: {FILTER_CONFIGS}")

# Apply sampling strategy
if SAMPLE_STRATEGY == "all":
    df_sample = df_filtered
    print(f"Using all {len(df_sample):,} records")

elif SAMPLE_STRATEGY == "random":
    n_sample = min(N_SAMPLES_TOTAL, len(df_filtered))
    df_sample = df_filtered.sample(n=n_sample, random_state=42)
    print(f"Random sample: {len(df_sample):,} records")

elif SAMPLE_STRATEGY == "per_model":
    samples = []
    for model in df_filtered['model_name'].unique():
        df_model = df_filtered[df_filtered['model_name'] == model]
        n_sample = min(N_SAMPLES_PER_MODEL, len(df_model))
        samples.append(df_model.sample(n=n_sample, random_state=42))
    df_sample = pd.concat(samples, ignore_index=True)
    print(f"Per-model sample: {len(df_sample):,} records ({N_SAMPLES_PER_MODEL} per model)")

elif SAMPLE_STRATEGY == "per_config":
    samples = []
    for config in df_filtered['source_config'].unique():
        df_config = df_filtered[df_filtered['source_config'] == config]
        n_sample = min(N_SAMPLES_PER_CONFIG, len(df_config))
        samples.append(df_config.sample(n=n_sample, random_state=42))
    df_sample = pd.concat(samples, ignore_index=True)
    print(f"Per-config sample: {len(df_sample):,} records ({N_SAMPLES_PER_CONFIG} per config)")

else:
    raise ValueError(f"Unknown sample strategy: {SAMPLE_STRATEGY}")

print(f"\n✅ Final dataset: {len(df_sample):,} records to evaluate")

# Show distribution
print(f"\n📊 Distribution:")
print("\nBy Model:")
print(df_sample['model_name'].value_counts())
print("\nBy Config:")
print(df_sample['source_config'].value_counts())
print("\nBy Ground Truth Type:")
print(df_sample['ground_truth_type'].value_counts())

## 5. Run Semantic Evaluation

In [ ]:
def evaluate_record(row: pd.Series) -> Dict:
    """
    Run semantic evaluation on a single record.
    
    Returns dict with evaluation results.
    """
    result = {
        'sample_id': row['sample_id'],
        'model_name': row['model_name'],
        'source_config': row['source_config'],
    }
    
    # Semantic F1 (Molmo-style)
    if ENABLE_SEMANTIC_F1:
        try:
            sem_scores = eval_utils.compute_semantic_f1(
                generated=row['response_raw'],
                ground_truth=row['ground_truth'],
                evaluator_port=GLIDER_PORT,
            )
            result.update({
                'semantic_precision': sem_scores['semantic_precision'],
                'semantic_recall': sem_scores['semantic_recall'],
                'semantic_f1': sem_scores['semantic_f1'],
            })
        except Exception as e:
            print(f"  ⚠️  Semantic F1 failed for {row['sample_id']}: {e}")
            result.update({
                'semantic_precision': None,
                'semantic_recall': None,
                'semantic_f1': None,
                'semantic_error': str(e),
            })
    
    # Glider rubric evaluation
    if ENABLE_GLIDER_RUBRIC:
        try:
            # Initialize evaluator for this call
            def glider_chat(messages, max_tokens, model_name):
                text, _ = eval_utils.make_vllm_request(
                    port=GLIDER_PORT,
                    model_id="PatronusAI/glider",
                    messages=messages,
                    max_tokens=max_tokens,
                    request_timeout=REQUEST_TIMEOUT,
                )
                return text or ""
            
            evaluator = GliderEvaluator(
                chat_fn=glider_chat,
                model_name="PatronusAI/glider",
            )
            
            glider_result = evaluator.evaluate(
                question=row['prompt_raw'],
                model_answer=row['response_raw'],
                ground_truth=row['ground_truth'],
                sample_id=row['sample_id'],
            )
            
            result.update({
                'glider_score': glider_result['score'],
                'glider_reasoning': glider_result['reasoning'],
                'glider_highlight': glider_result['highlight'],
                'glider_raw_output': glider_result['raw_output'],
            })
        except Exception as e:
            print(f"  ⚠️  Glider eval failed for {row['sample_id']}: {e}")
            result.update({
                'glider_score': None,
                'glider_reasoning': None,
                'glider_highlight': None,
                'glider_error': str(e),
            })
    
    return result


def save_checkpoint(results: List[Dict], batch_num: int):
    """Save intermediate results."""
    if not results:
        return
    
    df_checkpoint = pd.DataFrame(results)
    checkpoint_file = OUTPUT_DIR / f"checkpoint_batch_{batch_num:04d}.parquet"
    df_checkpoint.to_parquet(checkpoint_file)
    print(f"  💾 Saved checkpoint: {checkpoint_file.name}")

In [ ]:
# Run evaluation in batches
all_results = []
total_batches = (len(df_sample) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\n{'='*80}")
print(f"Starting semantic evaluation")
print(f"Total records: {len(df_sample):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Total batches: {total_batches}")
print(f"{'='*80}\n")

start_time = time.time()

with tqdm(total=len(df_sample), desc="Evaluating samples") as pbar:
    for batch_num in range(total_batches):
        start_idx = batch_num * BATCH_SIZE
        end_idx = min(start_idx + BATCH_SIZE, len(df_sample))
        batch_df = df_sample.iloc[start_idx:end_idx]
        
        print(f"\nBatch {batch_num + 1}/{total_batches} (records {start_idx}-{end_idx})")
        
        batch_results = []
        for idx, row in batch_df.iterrows():
            try:
                result = evaluate_record(row)
                batch_results.append(result)
                all_results.append(result)
            except Exception as e:
                print(f"  ❌ Failed to evaluate {row['sample_id']}: {e}")
            finally:
                pbar.update(1)
        
        # Save checkpoint after each batch
        if SAVE_INTERMEDIATE and batch_results:
            save_checkpoint(batch_results, batch_num)
        
        # Show progress
        elapsed = time.time() - start_time
        rate = len(all_results) / elapsed if elapsed > 0 else 0
        remaining = (len(df_sample) - len(all_results)) / rate if rate > 0 else 0
        print(f"  Progress: {len(all_results)}/{len(df_sample)} ({rate:.2f} rec/sec, ~{remaining/60:.1f}min remaining)")

elapsed_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"Evaluation complete!")
print(f"Time elapsed: {elapsed_time:.2f}s ({elapsed_time/60:.2f} min)")
print(f"Records processed: {len(all_results):,}")
print(f"Rate: {len(all_results)/elapsed_time:.2f} records/sec")
print(f"{'='*80}")

## 6. Merge Results

In [ ]:
# Convert results to DataFrame
df_eval = pd.DataFrame(all_results)

print(f"\n📊 Evaluation results: {len(df_eval):,} records")
print(f"\nColumns added:")
for col in df_eval.columns:
    if col not in ['sample_id', 'model_name', 'source_config']:
        print(f"  - {col}")

# Merge with original data
df_merged = df_sample.merge(
    df_eval,
    on=['sample_id', 'model_name', 'source_config'],
    how='left',
    suffixes=('', '_eval')
)

print(f"\n✅ Merged dataset: {len(df_merged):,} records")

# Show sample of results
print("\n📋 Sample results:")
display_cols = ['sample_id', 'model_name', 'source_config']
if ENABLE_SEMANTIC_F1:
    display_cols.extend(['semantic_precision', 'semantic_recall', 'semantic_f1'])
if ENABLE_GLIDER_RUBRIC:
    display_cols.append('glider_score')

print(df_merged[display_cols].head(10))

## 7. Analysis

In [ ]:
print("="*80)
print("SEMANTIC EVALUATION ANALYSIS")
print("="*80)

if ENABLE_SEMANTIC_F1:
    print("\n=== Semantic F1 Scores by Model ===")
    sem_by_model = df_merged.groupby('model_name')[[
        'semantic_precision', 'semantic_recall', 'semantic_f1'
    ]].agg(['mean', 'std', 'count'])
    print(sem_by_model)
    
    print("\n=== Semantic F1 Scores by Config ===")
    sem_by_config = df_merged.groupby('source_config')[[
        'semantic_precision', 'semantic_recall', 'semantic_f1'
    ]].mean().sort_values('semantic_f1', ascending=False)
    print(sem_by_config)

if ENABLE_GLIDER_RUBRIC:
    print("\n=== Glider Rubric Scores by Model ===")
    glider_by_model = df_merged.groupby('model_name')['glider_score'].agg([
        'mean', 'std', 'min', 'max', 'count'
    ]).sort_values('mean', ascending=False)
    print(glider_by_model)
    
    print("\n=== Glider Rubric Scores by Config ===")
    glider_by_config = df_merged.groupby('source_config')['glider_score'].mean().sort_values(ascending=False)
    print(glider_by_config)

# Correlation analysis
if ENABLE_SEMANTIC_F1 and ENABLE_GLIDER_RUBRIC:
    print("\n=== Correlation: Semantic F1 vs Glider Score ===")
    corr = df_merged[['semantic_f1', 'glider_score']].corr()
    print(corr)
    print(f"\nPearson correlation: {corr.loc['semantic_f1', 'glider_score']:.3f}")

# Compare with base metrics
if 'is_correct' in df_merged.columns:
    print("\n=== Comparison with Base Metrics ===")
    comparison = df_merged.groupby('model_name').agg({
        'is_correct': 'mean',
    })
    
    if ENABLE_SEMANTIC_F1:
        comparison['semantic_f1'] = df_merged.groupby('model_name')['semantic_f1'].mean()
    if ENABLE_GLIDER_RUBRIC:
        comparison['glider_score'] = df_merged.groupby('model_name')['glider_score'].mean()
    
    print(comparison.sort_values('is_correct', ascending=False))

## 8. Save Results

In [ ]:
# Save merged results
output_file = OUTPUT_DIR / f"semantic_results{OUTPUT_SUFFIX}.parquet"
df_merged.to_parquet(output_file, index=False)
print(f"✅ Saved merged results: {output_file}")

# Save evaluation-only results
eval_only_file = OUTPUT_DIR / f"semantic_scores{OUTPUT_SUFFIX}.parquet"
df_eval.to_parquet(eval_only_file, index=False)
print(f"✅ Saved evaluation scores: {eval_only_file}")

# Save summary statistics
summary = {
    'timestamp': datetime.now().isoformat(),
    'source_run_id': RUN_ID,
    'total_records_evaluated': len(df_eval),
    'semantic_f1_enabled': ENABLE_SEMANTIC_F1,
    'glider_rubric_enabled': ENABLE_GLIDER_RUBRIC,
    'sample_strategy': SAMPLE_STRATEGY,
    'elapsed_time_seconds': elapsed_time,
}

if ENABLE_SEMANTIC_F1:
    summary['semantic_f1_mean'] = float(df_merged['semantic_f1'].mean())
    summary['semantic_f1_by_model'] = df_merged.groupby('model_name')['semantic_f1'].mean().to_dict()

if ENABLE_GLIDER_RUBRIC:
    summary['glider_score_mean'] = float(df_merged['glider_score'].mean())
    summary['glider_score_by_model'] = df_merged.groupby('model_name')['glider_score'].mean().to_dict()

summary_file = OUTPUT_DIR / f"summary{OUTPUT_SUFFIX}.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✅ Saved summary: {summary_file}")

print(f"\n{'='*80}")
print(f"All results saved to: {OUTPUT_DIR}")
print(f"{'='*80}")

## 9. Inspect Individual Examples

In [ ]:
# Show best and worst performers
def show_examples(df: pd.DataFrame, metric: str, n: int = 3, ascending: bool = False):
    """
    Show top/bottom examples by a metric.
    """
    df_sorted = df.dropna(subset=[metric]).sort_values(metric, ascending=ascending)
    
    label = "Worst" if ascending else "Best"
    print(f"\n{'='*80}")
    print(f"{label} {n} examples by {metric}")
    print(f"{'='*80}")
    
    for i, (idx, row) in enumerate(df_sorted.head(n).iterrows(), 1):
        print(f"\n--- Example {i} ---")
        print(f"Model: {row['model_name']}")
        print(f"Config: {row['source_config']}")
        print(f"{metric}: {row[metric]:.3f}")
        print(f"\nPrompt: {row['prompt_raw'][:200]}...")
        print(f"\nGround Truth: {row['ground_truth'][:200]}...")
        print(f"\nModel Response: {row['response_raw'][:200]}...")
        
        if ENABLE_GLIDER_RUBRIC and 'glider_reasoning' in row:
            print(f"\nGlider Reasoning: {row['glider_reasoning']}")

# Show examples
if ENABLE_SEMANTIC_F1:
    show_examples(df_merged, 'semantic_f1', n=3, ascending=False)
    show_examples(df_merged, 'semantic_f1', n=3, ascending=True)

if ENABLE_GLIDER_RUBRIC:
    show_examples(df_merged, 'glider_score', n=3, ascending=False)
    show_examples(df_merged, 'glider_score', n=3, ascending=True)